# Simple: Remove Fields with PAMOLA.CORE

**Goal**: Learn field removal basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Remove specified fields using execute()
- Compare before/after results
- Understand memory and privacy benefits

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── transformations/remove_fields/
        └── 01_remove_fields_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.transformations.field_ops.remove_fields import RemoveFieldsOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run to load data from `examples/data_examples/sample.csv`
- Auto-creates sample data if file missing
- Review dataset structure before proceeding

**What you'll see:**
- Dataset summary (20 records, 8 columns)
- First 5 records with customer information
- Column statistics showing unique values and ranges
- Sensitive fields: name, email, phone, address (suitable for removal)

**Note:** Sample includes sensitive PII fields demonstrating privacy protection use case

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample customer data with sensitive fields
    sample_data = pd.DataFrame({
        'customer_id': range(1, 21),
        'name': [
            'John Doe', 'Jane Smith', 'Bob Johnson', 'Alice Williams',
            'Charlie Brown', 'Diana Davis', 'Eve Miller', 'Frank Wilson',
            'Grace Taylor', 'Henry Anderson', 'Ivy Thomas', 'Jack Jackson',
            'Kelly White', 'Liam Harris', 'Mia Martin', 'Noah Thompson',
            'Olivia Garcia', 'Paul Martinez', 'Quinn Robinson', 'Ryan Clark'
        ],
        'email': [
            'john.doe@email.com', 'jane.smith@email.com', 'bob.j@email.com',
            'alice.w@email.com', 'charlie.b@email.com', 'diana.d@email.com',
            'eve.m@email.com', 'frank.w@email.com', 'grace.t@email.com',
            'henry.a@email.com', 'ivy.t@email.com', 'jack.j@email.com',
            'kelly.w@email.com', 'liam.h@email.com', 'mia.m@email.com',
            'noah.t@email.com', 'olivia.g@email.com', 'paul.m@email.com',
            'quinn.r@email.com', 'ryan.c@email.com'
        ],
        'phone': [
            '555-0101', '555-0102', '555-0103', '555-0104', '555-0105',
            '555-0106', '555-0107', '555-0108', '555-0109', '555-0110',
            '555-0111', '555-0112', '555-0113', '555-0114', '555-0115',
            '555-0116', '555-0117', '555-0118', '555-0119', '555-0120'
        ],
        'address': [
            '123 Main St', '456 Oak Ave', '789 Pine Rd', '321 Elm St',
            '654 Maple Dr', '987 Cedar Ln', '147 Birch Way', '258 Spruce Ct',
            '369 Willow Pl', '741 Ash Blvd', '852 Poplar St', '963 Fir Ave',
            '159 Cherry Rd', '357 Beech Dr', '486 Hickory Ln', '264 Walnut Way',
            '375 Sycamore Ct', '486 Magnolia Pl', '597 Dogwood Blvd', '618 Redwood St'
        ],
        'city': [
            'New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix',
            'Philadelphia', 'San Antonio', 'San Diego', 'Dallas', 'San Jose',
            'Austin', 'Jacksonville', 'Fort Worth', 'Columbus', 'San Francisco',
            'Charlotte', 'Indianapolis', 'Seattle', 'Denver', 'Boston'
        ],
        'purchase_amount': [
            120.50, 89.99, 234.00, 156.75, 99.00, 310.25, 78.50, 445.00,
            167.80, 203.45, 91.20, 526.90, 134.65, 299.99, 412.30, 88.75,
            195.40, 267.85, 143.50, 389.60
        ],
        'membership_tier': [
            'Gold', 'Silver', 'Bronze', 'Gold', 'Silver', 'Platinum',
            'Bronze', 'Platinum', 'Gold', 'Silver', 'Bronze', 'Platinum',
            'Silver', 'Gold', 'Platinum', 'Bronze', 'Gold', 'Silver',
            'Bronze', 'Platinum'
        ]
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created")

df = read_csv(data_path)
print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:20s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
1. **CUSTOMIZE fields_to_remove** in `create_config_kwargs()`
   - Default: `["email", "phone"]`
   - Change to YOUR dataset's fields to remove
2. Run to validate configuration and setup environment

**What you'll see:**
- Fields to remove listed with unique value counts
- Field counts: total, to be removed, remaining after removal
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and progress tracker ready (✓)

**Note:** Removing sensitive fields improves privacy and reduces memory footprint

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "fields_to_remove": ["email", "phone"]  # Sensitive fields to remove
    }

kwargs = create_config_kwargs()
fields_to_remove = kwargs["fields_to_remove"]

# Validate fields exist
print(f"\n🔍 Validating field configuration...\n")
missing_fields = [f for f in fields_to_remove if f not in df.columns]
if missing_fields:
    raise ValueError(
        f"❌ Fields not found in dataset: {missing_fields}\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'fields_to_remove' in create_config_kwargs()"
    )

print(f"✓ Fields to remove: {fields_to_remove}")
print(f"  Total fields in dataset: {len(df.columns)}")
print(f"  Fields to be removed: {len(fields_to_remove)}")
print(f"  Fields remaining after removal: {len(df.columns) - len(fields_to_remove)}")
print(f"\n  Sensitive fields being removed:")
for field in fields_to_remove:
    print(f"    - {field}: {df[field].nunique()} unique values")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="field_removal",
    description=f"Removal of sensitive fields: {', '.join(fields_to_remove)}",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description="Processing field removal",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Review configuration parameters below
- Run to execute field removal
- Monitor progress bar (6 steps: load → validate → remove → save → complete)

**Key parameters:**
- `fields_to_remove`: List of field names to remove
- `pattern=None`: Optional regex pattern for field matching
- `output_format='csv'`: Output file format

**What you'll see:**
- Configuration summary with fields to remove
- Progress bar: load → validate fields → drop columns → save → complete
- Completion status: "completed" (verify no errors)
- Artifact count (3-5 files expected)

**Note:** Fields are permanently removed from output - original data unchanged in source

In [ ]:
# Create operation with production-style configuration
operation = RemoveFieldsOperation(
    # Core parameters
    fields_to_remove=fields_to_remove,
    pattern=None,  # Optional: regex pattern for field matching
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Fields to remove: {operation.fields_to_remove}")
print(f"  Pattern:          {operation.pattern or 'None'}")
print(f"  Visualizations:   {operation.generate_visualization}")
print(f"  Save output:      {operation.save_output}")
print(f"  Force recalc:     {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing field removal...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load processed data from task_dir
- Review field comparison and memory reduction
- Verify sensitive fields removed

**What you'll see:**
- Sample processed records (without removed fields)
- Field comparison: original vs remaining vs removed lists
- Memory reduction percentage and sizes (before/after)
- Summary: record counts, field counts, memory savings
- Key metrics: execution time, records processed

**Note:** Reduced field count improves privacy and lowers storage/memory requirements

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output file
output_files = list(task_dir.glob('output/*.csv'))
if output_files:
    output_file = output_files[0]
    result_df = pd.read_csv(output_file)
    
    print("\n" + "=" * 80)
    print("📊 RESULTS COMPARISON")
    print("=" * 80)
    
    # Show sample records
    print("\n🔍 Sample Processed Records:")
    print(result_df.head(10))
    
    # Field comparison
    print(f"\n📈 Field Comparison:")
    print(f"  Original fields:  {list(df.columns)}")
    print(f"  Remaining fields: {list(result_df.columns)}")
    print(f"  Removed fields:   {[f for f in df.columns if f not in result_df.columns]}")
    
    # Memory comparison
    original_memory = df.memory_usage(deep=True).sum()
    result_memory = result_df.memory_usage(deep=True).sum()
    memory_reduction = (original_memory - result_memory) / original_memory * 100
    
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    print(f"  Original records:   {len(df)}")
    print(f"  Processed records:  {len(result_df)}")
    print(f"  Original fields:    {len(df.columns)}")
    print(f"  Remaining fields:   {len(result_df.columns)}")
    print(f"  Fields removed:     {len(df.columns) - len(result_df.columns)}")
    print(f"  Memory reduction:   {memory_reduction:.1f}%")
    print(f"  Original memory:    {original_memory / 1024:.2f} KB")
    print(f"  Result memory:      {result_memory / 1024:.2f} KB")
    
    # Display result metrics
    if result.metrics:
        print("\n📊 Key Metrics:")
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 Fewer fields = Improved privacy & reduced storage!")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Run to display all generated files
- Navigate to directories for manual inspection
- Verify each folder has expected file count

**What you'll see:**
- Directory structure tree (output/, metrics/, visualizations/, config/)
- File count per directory (typically 1-2 files each)
- File names with sizes in KB
- Absolute path to task directory for manual navigation

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Processed CSV (fields removed)
├── metrics/          # Field removal and memory metrics JSON
├── visualizations/   # Field comparison charts (PNG)
└── config/           # Operation configuration
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['output', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Operation performance and effectiveness metrics
2. **Output Data**: Processed records with sample rows
3. **Visualizations**: Charts and graphs from the latest operation run

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST - with filtering)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))
    
    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # 1) Exclude data_types files
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]
        
        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)
        
        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract metadata and metrics
            metadata = data.get('metadata', {})
            metrics = data.get('metrics', {})
            
            # -------- METADATA --------
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")

            op = metadata.get("operation", {})
            print(
                f"   Operation: {op.get('class', 'N/A')}."
                f"{op.get('function', 'N/A')}"
            )

            # -------- FIELD REMOVAL METRICS --------
            print("\n📊 FIELD REMOVAL METRICS:")
            print("-" * 80)

            fields_removed_count = metrics.get("fields_removed_count")
            fields_removed_pct = metrics.get("fields_removed_percentage")

            if fields_removed_count is not None:
                print(f"   Fields Removed Count: {fields_removed_count}")

            if fields_removed_pct is not None:
                print(f"   Fields Removed %:     {fields_removed_pct:.2f}%")

            # -------- MEMORY USAGE --------
            mem = metrics.get("memory_usage_byte")
            if isinstance(mem, dict):
                before = mem.get("before")
                after = mem.get("after")

                print("\n💾 MEMORY USAGE:")
                print("-" * 80)

                if before is not None:
                    print(f"   Before: {before / 1024:.2f} KB")

                if after is not None:
                    print(f"   After:  {after / 1024:.2f} KB")

                if before and after and before > 0:
                    reduction = (before - after) / before * 100
                    print(f"   Reduction: {reduction:.2f}%")

            # -------- SHAPE COMPARISON --------
            shape = metrics.get("shape")
            if isinstance(shape, dict):
                print("\n📐 SHAPE COMPARISON:")
                print("-" * 80)
                print(f"   Before: {shape.get('before', 'N/A')}")
                print(f"   After:  {shape.get('after', 'N/A')}")

            # -------- PERFORMANCE --------
            print("\n⚡ PERFORMANCE:")
            print("-" * 80)

            exec_time = metrics.get("execution_time_seconds")
            if exec_time is not None:
                print(f"   Execution Time:     {exec_time:.4f}s")

            records = metrics.get("records_processed")
            if records is not None:
                print(f"   Records Processed:  {records:,}")

            rps = metrics.get("records_per_second")
            if rps is not None:
                print(f"   Records / Second:   {rps:.2f}")
            
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. OUTPUT DATA (NEWEST)
print("\n\n2️⃣ OUTPUT DATA (First 10 rows):")
print("-" * 80)

output_dir = task_dir / 'output'
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    output_files = list(output_dir.glob('*.csv'))
    
    if not output_files:
        print("⚠️  No output files found")
    else:
        # Sort by modification time (newest first)
        output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_output_file = output_files[0]
        
        # Show file info
        mtime = datetime.fromtimestamp(latest_output_file.stat().st_mtime)
        print(f"✓ Found {len(output_files)} output file(s)")
        print(f"📄 Loading LATEST: {latest_output_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_output_file.stat().st_size / 1024:.1f} KB\n")
        
        try:
            output_df = pd.read_csv(latest_output_file)
            print(f"📊 Shape: {output_df.shape[0]} rows × {output_df.shape[1]} columns")
            print(f"\n{output_df.head(10).to_string()}")
            
            # Show field comparison
            print(f"\n\n📊 Field Comparison:")
            print("-" * 80)
            print(f"Remaining fields: {list(output_df.columns)}")
            print(f"\nFields in output: {len(output_df.columns)}")
            
        except Exception as e:
            print(f"❌ Error reading output: {e}")

# 3. VISUALIZATIONS (NEWEST BATCH)
print("\n\n3️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**  
✅ Load data from examples/data_examples/  
✅ Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ Configure field removal with full parameters  
✅ Execute using operation.execute()  
✅ Analyze results and review artifacts  

**Key takeaways:**
- Remove sensitive fields to improve privacy
- Reduce dataset size and memory footprint
- Pattern-based field removal for flexibility
- All artifacts saved in structured directories

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_remove_fields_advanced.ipynb`** includes:
- Pattern-based field matching with regex
- Conditional field removal
- Multiple dataset processing
- Advanced memory optimization
- Production deployment patterns

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)